### Imports, preprocessing

In [35]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from datetime import datetime
from urllib.parse import urljoin
from contextlib import redirect_stderr
import json
import re
# Suppress warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import logging
# Suppress pdfminer logs
logging.getLogger("pdfminer").setLevel(logging.ERROR)

In [36]:
# Function to download and extract PDF content
def extract_pdf_content(pdf_url):
    try:
        # Download the PDF
        response = requests.get(pdf_url)
        with open("temp.pdf", "wb") as f:
            f.write(response.content)

        # Suppress warnings by redirecting stderr
        with open(os.devnull, "w") as devnull, redirect_stderr(devnull):
            with pdfplumber.open("temp.pdf") as pdf:
                pdf_content = "\n".join([page.extract_text() for page in pdf.pages])

        # Clean up the temporary file
        os.remove("temp.pdf")
        return pdf_content
    except Exception as e:
        print(f"Error extracting PDF content from {pdf_url}: {e}")
        return ""

In [37]:
def clean_content(text):
    # Step 1: Remove unwanted characters
    cleaned_text = re.sub(r'[\r\n]+', ' ', text) #Remove \n characters
    cleaned_text = re.sub(r'[^\x00-\x7F]+', '', text)  # Remove non-ASCII characters
    cleaned_text = re.sub(r'\u2013', '-', cleaned_text)   # Replace en-dash with hyphen
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)     # Collapse multiple spaces

    # Step 2: Remove page numbers and headers/footers
    cleaned_text = re.sub(r'\d+ \| PwC \| ', '', cleaned_text)
    cleaned_text = re.sub(r'Table of contents', '', cleaned_text)
    # Step 3: Remove extra whitespace and normalize formatting
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text

### Scraping

In [38]:
web_content_links = ["https://kpmg.com/us/en/media/news/harnessing-the-value-of-data.html",
"https://kpmg.com/us/en/media/news/kpmg-google-cloud-alliance-expansion-agentspace-adoption.html",
"https://kpmg.com/us/en/media/news/new-industrial-manufacturing-signals-to-watch-2025.html",
"https://kpmg.com/us/en/media/news/cfo-cio-partnership-innovation.html",
"https://kpmg.com/us/en/media/news/kpmg-risk-resilience-survey-2025.html",
"https://kpmg.com/us/en/articles/2024/cybersecurity-considerations-technology.html",
"https://kpmg.com/us/en/articles/2025/first-100-days-regulatory-signals-for-im-auto-reg-alert.html",
"https://kpmg.com/us/en/articles/2024/kpmg-intelligence-forecasting-use-cases.html"]

In [39]:
def scrape_and_save_data(urls):
    all_data = []
    for url in urls:
        try:
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "No Title"

            # Extract date from HTML
            date_element = soup.select_one('.cmp-herobasic__date span')
            if date_element:
                raw_date = date_element.get_text(strip=True)
                try:
                    parsed_date = datetime.strptime(raw_date, "%B %d, %Y")  # Parses "April 10, 2025"
                    date_published = parsed_date.strftime("%Y-%m-%d")        # Converts to "2025-04-10"
                except ValueError:
                    date_published = raw_date  # Fallback to original string if parsing fails
            else:
                date_published = "No Date"


            content = "\n".join([p.get_text(strip=True) for p in soup.find_all('p')])
            content = clean_content(content)

            download_button = soup.find('a', class_='cmp-pdfdownload__download-cta')
            pdf_content = ""

            if download_button and 'href' in download_button.attrs:
                pdf_url = urljoin(url, download_button['href'])
                print(f"Downloading PDF from: {pdf_url}")
                pdf_content = extract_pdf_content(pdf_url)
                pdf_content = clean_content(pdf_content)

            article_data = {
                "url": url,
                "title": title,
                "date": date_published,
                "content": content,
                "pdf_content": pdf_content
            }
            all_data.append(article_data)

        except requests.exceptions.RequestException as e:
            print(f"Error fetching URL {url}: {e}")
            all_data.append({"url": url, "error": str(e)}) # Store error info
        except Exception as e:
            print(f"An error occurred processing {url}: {e}")
            all_data.append({"url": url, "error": str(e)}) #Store error info

    # Save to JSON file
    try:
        with open('scraped_data_kpmg.json', 'w') as f:
            json.dump(all_data, f, indent=4)
        print("Data saved to scraped_data_kpmg.json")
    except Exception as e:
        print(f"Error saving to JSON file: {e}")

scrape_and_save_data(web_content_links)


Data saved to scraped_data_kpmg.json
